# Example submission — random baseline (full NDIF round-trip)

A minimal submission that wires up the **whole pipeline** end to end: it reads the
eval set and, for every row, loads the model that generated it (base + LoRA) and
runs one remote **NDIF** trace over the conversation — then predicts a random coin
flip. It scores ~0.5 AUROC on purpose; replace the prediction with a real signal
read off the trace (see [`../tutorials/2-predicting.ipynb`](../tutorials/2-predicting.ipynb)
for an nnsight + NDIF probe).

**Contract:** running this top-to-bottom must write `submission.csv` with columns
`id,prediction` — one row per example, `id` set to the row's `index`, and
`prediction` a probability in `[0, 1]`.

In [ ]:
import os

# ── DO NOT CHANGE ───────────────────────────────────────────────────
# The runner sets these for each run; read them, don't hard-code or override.
#   DATASET_NAME : the eval dataset to predict on.
#   NDIF_HOST    : the NDIF cluster nnsight traces run on (set automatically;
#                  remote=True traces use it — you don't configure a host).
#   NDIF_API_KEY / HF_TOKEN : your forwarded credentials (already in the env).
DATASET_NAME = os.environ["DATASET_NAME"]

# The output file the grader reads — keep this name.
SUBMISSION_PATH = "submission.csv"

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from datasets import load_dataset

examples = load_dataset(DATASET_NAME, split="test")
print(f"loaded {len(examples)} examples")

# Each row is one example to score. The columns:
#   index    : stable row id. Put THIS in submission.csv's `id` so the grader can
#              line your prediction up with the held-out label — use it, not the
#              row's position (the two don't always match).
#   model    : the model that produced this conversation. Load it (next cell) to
#              read its internals.
#   lora     : LoRA adapter to attach when loading `model` (None for a base model).
#   messages : the conversation — a list of {"role", "content"} turns (system /
#              user / assistant). This is the text your detector judges.
print("columns:", examples.column_names)
print("first row (index/model/lora):",
      {k: examples[0][k] for k in ("index", "model", "lora")})

In [ ]:
from nnsight import VisionLanguageModel

# Load one model up front: grab it from the first row and load it a single time,
# then reuse it for every trace below. peft=<lora> attaches the LoRA adapter; lora
# is None for a base model. These eval sets are all gemma-3-27b-it, which is
# multimodal, so we use VisionLanguageModel (LanguageModel rejects multimodal
# repos). The weights live on NDIF — nothing heavy downloads here.
first = examples[0]
model = VisionLanguageModel(first["model"], peft=first["lora"])

In [ ]:
import torch

# Batched 50/50 baseline that still exercises the whole remote path end to end.
# The NDIF GPUs are SHARED and the 27B model nearly fills them, so we use a fixed
# batch size picked to survive a batch of the LONGEST (truncated) prompts — any
# batch of shorter prompts then fits too. Batch memory is dominated by attention
# (~rows x seqlen^2), so we also cap the sequence length.
MAX_LEN    = 256     # truncate to the LAST 256 tokens (keep the response — that's
                     #   where the lie is); a batch of these traces safely.
BATCH_SIZE = 8       # largest batch of full-length (MAX_LEN) prompts that fits.

rng = np.random.default_rng(0)
tok = model.tokenizer
tok.truncation_side = "left"          # drop the START, keep the assistant's answer
pad_id = tok.pad_token_id

def to_ids(messages):
    """Render the {"role","content"} turns with the model's chat template (matching
    what the model saw at generation time), then tokenize (truncated to MAX_LEN)."""
    if getattr(tok, "chat_template", None):
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    else:
        text = "\n".join(f"{m['role']}: {m['content']}" for m in messages)
    return tok(text, add_special_tokens=False, truncation=True, max_length=MAX_LEN)["input_ids"]

# Tokenize every row (keep its original position) and sort short->long, so most
# batches pad to a short length and trace fast.
encoded = sorted(((i, to_ids(ex["messages"])) for i, ex in enumerate(examples)),
                 key=lambda t: len(t[1]))

predictions = np.empty(len(examples), dtype=float)
ids = [ex["index"] for ex in examples]      # align by the row's `index`, not its position
n_batches = (len(encoded) + BATCH_SIZE - 1) // BATCH_SIZE

for bi in range(n_batches):
    chunk = encoded[bi * BATCH_SIZE:(bi + 1) * BATCH_SIZE]
    positions = [p for p, _ in chunk]
    seqs = [s for _, s in chunk]
    width = max(len(s) for s in seqs)
    # Left-pad to the batch's longest sequence (padding_side='left' => every row's
    # real last token is at index -1).
    input_ids = torch.tensor([[pad_id] * (width - len(s)) + s for s in seqs])
    attn      = torch.tensor([[0] * (width - len(s)) + [1] * len(s) for s in seqs])

    # One remote forward pass on NDIF over the whole batch.
    with model.trace({"input_ids": input_ids, "attention_mask": attn}, remote=True):
        feats = model.model.language_model.layers[-1].output[:, -1, :].save()  # (rows, hidden)

    # ── this is where your method would go ──────────────────────────────────────
    # `feats` is one (hidden,) activation vector per row — feed it to your detector
    # to get P(deceptive). The baseline ignores it and flips a coin.
    for pos in positions:
        predictions[pos] = float(rng.random())     # random 50/50
    print(f"  batch {bi + 1}/{n_batches}", end="\r")
print()

predictions = np.clip(predictions, 0.0, 1.0)

In [ ]:
assert len(predictions) == len(examples), "need exactly one prediction per example"

# `id` is each row's `index` (collected alongside the predictions above) — that's
# the key the grader joins your predictions to the held-out labels on.
pd.DataFrame({"id": ids, "prediction": predictions}).to_csv(
    SUBMISSION_PATH, index=False)
print(f"wrote {len(predictions)} predictions to {SUBMISSION_PATH}")